# Advanced Production RAG

Hybrid retrieval, RRF, reranking, HyDE, GraphRAG, and evals on live production traces.

This notebook is an original tutorial extending notebook 14 with current
(2026) production practice — the exact stack a serious RAG deployment ships:
*query rewrite → hybrid retrieval → RRF → rerank → generate with citations*,
instrumented with tracing and online LLM-as-judge evals.

## Learning Objectives

- Explain when BM25 beats dense retrieval and vice versa, and why their failure sets are disjoint.
- Derive and implement RRF, and explain why rank fusion beats score normalization.
- Describe the two-stage retrieve-then-rerank funnel and cross-encoder vs bi-encoder mechanics.
- Give the honest 2026 status of HyDE and what displaced it.
- Explain the GraphRAG pipeline, when graph beats vector, and the LazyGraphRAG cost fix.
- Describe production RAG evaluation: RAGAS metrics, LangSmith run trees, online judges on live traces.

## Mental Model

Production RAG is a **funnel of increasingly expensive, increasingly accurate
scorers**:

```
query rewrite (small LLM)
   ▼
hybrid retrieval: BM25  +  dense vector      (cheap, high recall, disjoint failures)
   ▼
RRF fusion (rank-based, tuning-free)          (rewards consensus)
   ▼
cross-encoder rerank top-50 → top-5           (expensive, high precision)
   ▼
generate with citations
```

Each stage exists because the previous one is cheap-but-lossy. Special query
classes get routed around the funnel: vocabulary-mismatch queries to
HyDE-style expansion, multi-hop/global questions to a graph layer.

## Important Concepts

- **BM25**: lexical scoring — wins on exact identifiers (error codes, SKUs, function names, acronyms). Dense wins on paraphrase and cross-lingual. Hybrid adds ~9 MRR points over semantic-only in 2026 benchmarks.
- **RRF**: `score(d) = sum over retrievers of 1/(k + rank(d))`, k=60 (Cormack et al., 2009). Rank-based, distribution-free, rewards consensus. Weighted variant: `w_r/(k + rank_r)`.
- **Cross-encoder reranking**: `[query, doc]` through one transformer with full cross-attention — far more accurate than bi-encoders, O(n) per query, so only rescores a small candidate set. Cohere Rerank, Voyage rerank, bge-reranker-v2-m3 (self-host, ~50-100ms GPU).
- **HyDE**: LLM writes a hypothetical answer; embed *that* for search — closes query↔document asymmetry. 2026 status: gated/selective, largely displaced by better instruction-tuned embeddings and document-side fixes.
- **Contextual retrieval** (Anthropic): prepend 50-100 tokens of LLM-written situating context to each chunk before embedding *and* BM25 indexing — -49% retrieval failures; -67% with reranking; ~$1.02/M doc tokens one-time.
- **GraphRAG**: entities/relations → knowledge graph → Leiden communities → community summaries → local search (entity-anchored) and global search (map-reduce over summaries). **LazyGraphRAG**: NLP-only indexing (~0.1% of the cost), LLM work deferred to query time.
- **RAGAS**: faithfulness + answer relevancy (diagnose generation); context precision + recall (diagnose retrieval).

## Need-To-Know Coverage Checklist

- [x] BM25 formula shape and when lexical wins; disjoint failure sets.
- [x] RRF formula, k=60, why rank beats score normalization, weighted RRF, the pgvector two-CTE SQL pattern.
- [x] Bi-encoder vs cross-encoder; retrieve top-50-100 → rerank → top-3-5.
- [x] Current rerankers and latency/cost; reranking as the highest-ROI lever after hybrid.
- [x] HyDE mechanism, when it helps/hurts, honest 2026 status; HyPE (index-time hypothetical questions).
- [x] GraphRAG pipeline (Leiden, community summaries, local vs global search); LazyGraphRAG economics; when graph beats vector.
- [x] RAGAS metric definitions; hit rate, MRR, NDCG.
- [x] LangSmith run trees, online evaluators on sampled prod traces, the trace→dataset flywheel.
- [x] Query rewriting, contextual retrieval, late chunking, ColBERT/ColPali, agentic RAG, semantic caching.

## Deep Study Notes

### Hybrid retrieval and RRF

- BM25 and dense retrieval fail on *different* queries — that disjointness is
  the whole argument for hybrid. BM25: exact tokens (identifiers, rare terms).
  Dense: meaning (paraphrase, synonymy, cross-lingual).
- **BM25's shape** (know the three ingredients, not the constants): score is a
  sum over query terms of `IDF(term) x term-frequency saturation x length
  normalization`. IDF upweights rare terms; the saturation term (`k1`≈1.2)
  means the 10th occurrence of a word adds far less than the 2nd; length
  normalization (`b`≈0.75) stops long documents winning just by being long.
- **Why RRF over score fusion**: BM25 scores are unbounded and
  corpus-dependent; cosine similarities cluster in a narrow band; the
  distributions are incomparable and shift per query. Min-max normalization is
  unstable (one outlier crushes the rest). RRF uses only ordinal rank —
  distribution-free, no calibration, no training data.
- The standard criticism (and your comeback): RRF discards score *magnitude* —
  a runaway-best match and a marginal-best match look identical. Answer: the
  cross-encoder reranker downstream restores fine-grained scoring.
- Built in: Elasticsearch (`rrf` retriever, weighted), OpenSearch, Qdrant
  (server-side since 1.10), Azure AI Search. **pgvector**: no native RRF —
  two CTEs (full-text + vector), `ROW_NUMBER()` each, FULL OUTER JOIN,
  `COALESCE(1.0/(60+rank_a),0)+COALESCE(1.0/(60+rank_b),0)`.

### Reranking

- Bi-encoder compresses each doc into one vector *before* seeing the query —
  fast, precomputable, lossy. Cross-encoder attends across query and document
  tokens jointly — accurate, nothing precomputable.
- Funnel: retrieve ~50-100 (practitioners increasingly ~20) → rerank → send
  3-5 to the LLM. Adds ~100-600ms.
- Models: Cohere Rerank 3.5/4 (managed default), Voyage rerank-2.5 (domain
  variants worth +2-4 NDCG in-domain), bge-reranker-v2-m3 (Apache 2.0,
  self-host, ~50-100ms on GPU), LLM-as-reranker (most accurate, offline only).
- The number to quote: Anthropic's contextual-retrieval study — reranking took
  retrieval-failure reduction from 49% to **67%**.

### HyDE — the honest answer

- Mechanism: doc-to-doc similarity is easier than query-to-doc; the fake
  answer lives in the same embedding neighborhood as real answers.
- Helps: zero-shot domains, short/underspecified queries, vocabulary mismatch.
  Hurts: hallucinated wrong-topic answers retrieve confidently wrong docs;
  keyword-precise queries get diluted. Costs one LLM generation per query
  (+300-1000ms).
- 2026 status: *still legitimate, no longer default.* Displaced by (a)
  instruction-tuned embeddings handling query/doc asymmetry natively, (b)
  cheap query rewriting, (c) document-side fixes — contextual retrieval and
  HyPE (precompute at index time the questions each chunk answers). The
  senior-sounding line: "we ship HyDE selectively behind a query classifier
  and prefer document-side augmentation for hot paths."

### GraphRAG

- Pipeline: LLM extracts entities + relations per chunk → knowledge graph →
  **Leiden** hierarchical community detection → LLM writes a summary per
  community per level → query time: **local search** (entity-anchored, fan
  out to neighbors) or **global search** (map-reduce over community summaries
  — answers "what are the main themes across the corpus?", which vector RAG
  structurally cannot, since no single chunk contains the answer).
- The #1 objection is indexing cost: an LLM call per chunk plus per community,
  re-incurred on updates. **LazyGraphRAG** answers it: NLP-only noun-phrase/
  co-occurrence indexing (~0.1% of the cost, ≈ vector RAG), LLM work deferred
  to query time with a tunable budget; beats comparisons at ~4% of global
  search's query cost.
- Graph beats vector on: multi-hop traversal questions, global sensemaking,
  entity disambiguation, relationships that never co-occur in one chunk.
  Lighter alternatives: LightRAG, Neo4j-as-filter around vector hits,
  HippoRAG. Pragmatic stance: graph layer only for the query classes that
  need it, routed by a classifier.

### Production evaluation

- **RAGAS**: *faithfulness* (decompose answer into atomic claims; fraction
  supported by retrieved context — the hallucination detector), *answer
  relevancy* (questions generated from the answer, cosine vs original),
  *context precision* (relevant chunks ranked at top?), *context recall*
  (ground-truth claims covered by retrieval?). The split localizes failures:
  faithfulness/relevancy → generation; precision/recall → retrieval.
- Offline retrieval metrics: hit rate/recall@k, MRR (mean of 1/rank of first
  relevant), NDCG@k (rank- and grade-aware; standard for reranker comparisons).
- **LangSmith on live traces**: every request is a run tree (retriever span
  with returned docs, prompt construction, LLM call with tokens/cost). Online
  evaluators attach LLM-as-judge to a sampled percentage of production traces
  (no ground truth needed); scores land as feedback, chartable/alertable.
  The flywheel: filter low-scoring traces → annotate → dataset → offline
  regression suite for the next change. Caveats to volunteer: judge bias,
  align the judge with human labels first, sample 5-10% for cost.

### The rest of the 2026 toolbox

Query rewriting/decomposition (de-reference conversation context, split
multi-part questions), contextual retrieval, late chunking (embed whole doc,
pool chunk vectors after — whole-doc context, no LLM cost), ColBERT/late
interaction (per-token embeddings, MaxSim — near-cross-encoder quality,
precomputable; ColPali for PDF/table-heavy corpora), agentic RAG (retrieval
as a tool in a loop; router sends easy queries to the single-shot pipeline,
hard ones to the loop), semantic caching (cache by query-embedding
similarity, ~0.95 threshold; 20-40% hit rates on FAQ traffic; beware
near-but-different queries like "cancel order" vs "cancel subscription").

## Common Failure Modes

- Score-normalized fusion breaking per-query → unstable rankings; use RRF.
- Reranking 500 candidates → latency blowup for negligible quality gain.
- HyDE on keyword-precise queries → exact identifiers diluted, worse than BM25 alone.
- Full GraphRAG on a frequently-updated corpus → indexing cost re-incurred forever.
- Evaluating only end-to-end answer quality → can't tell if retrieval or generation failed (this is what the RAGAS split is for).
- Semantic cache serving a near-but-different answer → correctness bug shipped as a latency optimization.

## Implementation Notes

- Default stack: query rewrite → contextual BM25 + contextual dense → RRF(k=60) → rerank top-50→5 → generate with citations.
- Route, don't stack: HyDE and GraphRAG behind a query classifier, not in every request.
- Instrument retrieval spans (which docs, which ranks) from day one — you cannot debug RAG without seeing what was retrieved.
- Rerankers are the highest-ROI upgrade to an existing pipeline; try bge-reranker-v2-m3 before paying for an API.

In [ ]:
"""Hybrid retrieval with RRF fusion and a toy reranker — the production
funnel in miniature. Structure identical to the real thing; swap the toy
scorers for BM25/pgvector/a cross-encoder.
"""
DOCS = {
    "d1": "Reset your password from the account settings page.",
    "d2": "Error E-4012 means the authentication token expired.",
    "d3": "Login failures are usually caused by expired sessions.",
    "d4": "Billing cycles renew on the first of each month.",
    "d5": "If you cannot sign in, your session token may have lapsed.",
}
QUERY = "user cannot log in, error E-4012"

# --- two retrievers with disjoint strengths -------------------------------
def lexical_rank(query, docs):
    """BM25 stand-in: exact token overlap. Nails 'E-4012'."""
    q = set(query.lower().replace(",", "").split())
    scores = {d: len(q & set(t.lower().replace(".", "").split())) for d, t in docs.items()}
    return [d for d, s in sorted(scores.items(), key=lambda x: -x[1]) if s > 0]

def dense_rank(query, docs):
    """Embedding stand-in: hand-set semantic ranking. Nails paraphrase
    ('cannot log in' ~ 'sign in', 'login failures') but misses 'E-4012'."""
    return ["d3", "d5", "d1", "d2"]

def rrf(rankings, k=60, weights=None):
    weights = weights or [1.0] * len(rankings)
    scores = {}
    for w, ranking in zip(weights, rankings):
        for rank, doc in enumerate(ranking, start=1):
            scores[doc] = scores.get(doc, 0.0) + w / (k + rank)
    return sorted(scores.items(), key=lambda x: -x[1])

lex, den = lexical_rank(QUERY, DOCS), dense_rank(QUERY, DOCS)
fused = rrf([lex, den])
print("lexical :", lex)
print("dense   :", den)
print("RRF     :", [(d, round(s, 4)) for d, s in fused])
# d2 (exact error code) and d3/d5 (paraphrase) all surface — consensus wins.

# --- rerank stage: expensive scorer on the small fused candidate set ------
def rerank(query, candidates, top_n=2):
    """Cross-encoder stand-in: joint relevance judgment on [query, doc]."""
    relevance = {"d2": 0.95, "d5": 0.80, "d3": 0.75, "d1": 0.40, "d4": 0.05}
    return sorted(candidates, key=lambda d: -relevance[d])[:top_n]

final = rerank(QUERY, [d for d, _ in fused[:4]])
print("rerank  :", final, "-> sent to the LLM with citations")


## Practice

1. Add weighted RRF (weight lexical 2x) and observe how the ranking shifts.
   When would you actually do this? (Hint: code search.)
2. Write the pgvector RRF pattern as SQL pseudocode: two CTEs, ROW_NUMBER,
   FULL OUTER JOIN, COALESCE.
3. Compute faithfulness by hand: an answer makes 4 claims, 3 are supported by
   retrieved context. Now explain what a low faithfulness + high context
   recall combination tells you about where the failure is.
4. A teammate proposes HyDE for every query. Give the three-part 2026 answer
   (better embeddings, cheaper rewriting, document-side fixes) and the
   selective-gating compromise.
5. Your corpus is 10-K filings and the top user question is "summarize the
   main risks across all filings." Which retrieval architecture and which
   GraphRAG search mode, and why can't vector RAG answer it?

## Design Checklist

- [ ] Hybrid (lexical + dense) with RRF, not score fusion.
- [ ] Contextual chunk augmentation applied to both indexes.
- [ ] Cross-encoder rerank stage; candidates capped (~50) for latency.
- [ ] Query classifier routes vocabulary-mismatch and multi-hop/global queries to HyDE/graph paths.
- [ ] Retrieval spans traced (docs, ranks, scores) on every request.
- [ ] RAGAS-style split metrics: retrieval vs generation failures separable.
- [ ] Online judge on sampled prod traces; low-scoring traces flow into the regression dataset.
- [ ] Semantic cache (if any) has TTLs, tenant scoping, and correctness evals.